In [ ]:
# 自动安装依赖库
import sys
import subprocess

def install_packages():
    packages = ['pillow', 'requests', 'ipywidgets', 'matplotlib']
    for package in packages:
        try:
            __import__(package.replace('-', '_'))
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_packages()

# 导入必要的库
import base64
import io
import json
import requests
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from io import BytesIO
import time

# 全局变量
picture1 = None
picture2 = None
resultImage = None

# API选项
API_OPTIONS = {

    "Pollinations.ai (Free, No Key)": "pollinations",

}

# 创建输出区域
output = widgets.Output()

# 从URL或本地路径加载图片
def load_image(path):
    """
    加载图片，支持URL和本地路径
    """
    try:
        if path.startswith('http://') or path.startswith('https://'):
            response = requests.get(path)
            img = Image.open(BytesIO(response.content))
        else:
            img = Image.open(path)

        img = img.resize((512, 512))
        return img
    except Exception as e:
        print(f"Error loading image from {path}: {e}")
        return None

def image_to_base64(img):
    """将PIL Image转换为base64字符串"""
    buffered = BytesIO()
    img.save(buffered, format="JPEG", quality=100)
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return img_str

def base64_to_image(b64_string):
    """将base64字符串转换为PIL Image"""
    img_data = base64.b64decode(b64_string)
    img = Image.open(BytesIO(img_data))
    return img

def display_images():
    """显示所有图片"""
    with output:
        clear_output(wait=True)

        num_images = sum([picture1 is not None, picture2 is not None, resultImage is not None])

        if num_images == 0:
            print("No images to display")
            return

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        if picture1:
            axes[0].imshow(picture1)
            axes[0].set_title('Picture 1')
            axes[0].axis('off')
        else:
            axes[0].axis('off')

        if picture2:
            axes[1].imshow(picture2)
            axes[1].set_title('Picture 2')
            axes[1].axis('off')
        else:
            axes[1].axis('off')

        if resultImage:
            axes[2].imshow(resultImage)
            axes[2].set_title('Result')
            axes[2].axis('off')
        else:
            axes[2].axis('off')

        plt.tight_layout()
        plt.show()

# ====== API 实现 ======

def generate_with_pollinations(prompt):
    """
    使用 Pollinations.ai - 完全免费，无需API Key
    """
    try:
        # Pollinations.ai 的简单URL方式
        encoded_prompt = requests.utils.quote(prompt)
        url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=512&height=512&nologo=true"

        with status_output:
            clear_output()
            print(f"🎨 Generating with Pollinations.ai...")
            print(f"Prompt: {prompt}")

        response = requests.get(url, timeout=60)

        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))
            return img
        else:
            raise Exception(f"Status code: {response.status_code}")

    except Exception as e:
        raise Exception(f"Pollinations API error: {e}")



# 主要生成函数
def ask(button):
    """发送请求生成图片"""
    global resultImage

    try:
        selected_api = api_selector.value
        prompt = input_box.value
        api_key = api_key_input.value if api_key_input.value else None

        if not prompt:
            with status_output:
                clear_output()
                print("✗ Please enter a prompt!")
            return

        # 根据选择的API生成图片
        if selected_api == "pollinations":
            resultImage = generate_with_pollinations(prompt)


        with status_output:
            clear_output()
            print("✓ Image generated successfully!")

        display_images()

    except Exception as e:
        with status_output:
            clear_output()
            print(f"✗ Error: {e}")
            import traceback
            traceback.print_exc()

def load_pictures(button):
    """加载两张图片"""
    global picture1, picture2

    with status_output:
        clear_output()
        print("Loading images...")

    picture1 = load_image(image1_path.value)
    picture2 = load_image(image2_path.value)

    if picture1 and picture2:
        with status_output:
            clear_output()
            print("✓ Images loaded successfully!")
        display_images()
    else:
        with status_output:
            clear_output()
            print("✗ Failed to load images. Please check the paths.")

# ====== UI 组件 ======

# 图片路径输入框
image1_path = widgets.Text(
    value='https://picsum.photos/512/512?random=1',
    placeholder='Enter image 1 path or URL',
    description='Image 1:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

image2_path = widgets.Text(
    value='https://picsum.photos/512/512?random=2',
    placeholder='Enter image 2 path or URL',
    description='Image 2:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

load_button = widgets.Button(
    description='Load Images',
    button_style='info',
    icon='download'
)
load_button.on_click(load_pictures)

# 提示词输入框
input_box = widgets.Textarea(
    value='A beautiful landscape with mountains and a lake at sunset, digital art, highly detailed',
    placeholder='Enter your prompt',
    description='Prompt:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px', height='80px')
)

# 生成按钮
ask_button = widgets.Button(
    description='Generate Image',
    button_style='success',
    icon='check'
)
ask_button.on_click(ask)

# 状态输出
status_output = widgets.Output()

# 布局
ui = widgets.VBox([
    widgets.HTML("<h2>🎨 AI Image Generation</h2>"),



    widgets.HTML("<h3>📸 Load Images</h3>"),
    image1_path,
    image2_path,
    load_button,

    widgets.HTML("<h3>✨ Generate Image</h3>"),
    input_box,
    ask_button,

    widgets.HTML("<h3>📊 Status</h3>"),
    status_output,

    widgets.HTML("<h3>🖼️ Results</h3>"),
    output
])

display(ui)

